# Data prep: cleaning and rebuilding coordinates

BTS Consumer Airfare Report, Table 1 (1996 Q1 to 2025 Q4). Top 1,000 U.S. city-pair markets.

Written in Google Colab. Raw files were read from a shared Drive folder, so update the paths to run locally.

### Key columns

- **Year, quarter**: when the data was recorded
- **citymarketid_1/2, city1/2**: the two city markets on the route
- **nsmiles**: distance in miles
- **passengers**: average passengers per day
- **fare**: average fare on the route
- **carrier_lg, large_ms, fare_lg**: the airline with the most passengers, its share, its fare
- **carrier_low, lf_ms, fare_low**: the airline with the lowest fare, its share, its fare
- **Geocoded_City1/2** and related columns: coordinates and address fields (dropped in Step 2)

## Step 1. Load data and check


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd

In [3]:
PATH = "/content/drive/MyDrive/770 EDA_Team 9/Airfare_Table1.csv"

In [4]:
table1 = pd.read_csv(PATH, dtype=str)

In [5]:
table1.shape

(120035, 26)

In [6]:
table1.head()

,Year,quarter,citymarketid_1,citymarketid_2,city1,city2,nsmiles,passengers,fare,carrier_lg,...,Geocoded_City1,Geocoded_City1 (address),Geocoded_City1 (city),Geocoded_City1 (state),Geocoded_City1 (zip),Geocoded_City2,Geocoded_City2 (address),Geocoded_City2 (city),Geocoded_City2 (state),Geocoded_City2 (zip)
0,2025,4,"32,467",31703,"Miami, FL (Metropolitan Area)","New York City, NY (Metropolitan Area)",1118,"17,757",$208.11,DL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,4,"32,575",32457,"Los Angeles, CA (Metropolitan Area)","San Francisco, CA (Metropolitan Area)",379,"16,680",$178.28,WN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,4,"32,575",31703,"Los Angeles, CA (Metropolitan Area)","New York City, NY (Metropolitan Area)",2510,"12,978",$464.03,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,4,"31,703",31454,"New York City, NY (Metropolitan Area)","Orlando, FL",989,"12,914",$173.47,B6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,4,"30,977",31703,"Chicago, IL","New York City, NY (Metropolitan Area)",773,"10,926",$206.29,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Blanks per column

In [9]:
table1.isna().sum()

,0
Year,0
quarter,0
citymarketid_1,0
citymarketid_2,0
city1,0
city2,0
nsmiles,0
passengers,0
fare,0
carrier_lg,0


## Step 2. Validate the geocode columns

### Build a city-level geocode table

In [12]:
# cities appear in both city1 and city2, so stack the two columns
# reorganize city 1 info
a = table1[["city1", "Geocoded_City1"]].rename(
    columns={"city1": "city", "Geocoded_City1": "geo"}
)
# reorganize city 2 info
b = table1[["city2", "Geocoded_City2"]].rename(
    columns={"city2": "city", "Geocoded_City2": "geo"}
)
# stack the two cities and get rid of duplicates
geo = pd.concat([a, b]).dropna().drop_duplicates().reset_index(drop=True)

In [13]:
# how many distinct cities exist across both columns
all_cities = set(table1["city1"]) | set(table1["city2"])
len(all_cities)

178

In [14]:
# POINT (lon lat) -> two numeric columns, lon comes first
geo[["lon", "lat"]] = (
    geo["geo"].str.extract(r"POINT \(([-\d.]+) ([-\d.]+)\)").astype(float)
)
# 178 cities total, so anything under that means missing coordinates
print(len(geo), "cities have a geocode")
geo.head()

167 cities have a geocode


,city,geo,lon,lat
0,"Miami, FL (Metropolitan Area)",POINT (-93.264346 44.977479),-93.264346,44.977479
1,"Los Angeles, CA (Metropolitan Area)",POINT (-118.243344 34.052238),-118.243344,34.052238
2,"Dallas/Fort Worth, TX",POINT (-77.035636 40.11086),-77.035636,40.110860
3,"Chicago, IL",POINT (-87.696388 41.775002),-87.696388,41.775002
4,"Atlanta, GA (Metropolitan Area)",POINT (-84.391502 33.748547),-84.391502,33.748547


In [15]:
# which cities have no geocode at all
sorted(all_cities - set(geo["city"]))

['Ashland, WV',
 'Charlottesville, VA',
 'Hilton Head, SC',
 'Montrose/Delta, CO',
 'New Haven, CT',
 'Quad Cities, IL (Metropolitan Area)',
 'Springfield, IL',
 'St. Cloud, MN',
 'Traverse City, MI',
 'Vero Beach, FL',
 'Worcester, MA']

### Why all 10 geocode columns get dropped

The coordinates are wrong or missing:
1. 26 cities share identical coordinates with another city
2. Coverage drops to half in 2019-2020 and to zero from 2021
3. 11 of 178 cities never had coordinates

The other geocode columns add nothing:
4. 6 columns are completely blank, and 2 just repeat city1/city2

**1. Do different cities share the same coordinates?** Yes, 26 of them.

In [16]:
dupes = geo[geo.duplicated("geo", keep=False)].sort_values("geo")
print(len(dupes), " Duplicate geocode cities count")
dupes[["city", "lon", "lat"]]

26  Duplicate geocode cities count


,city,lon,lat
1,"Los Angeles, CA (Metropolitan Area)",-118.243344,34.052238
83,"Lexington, KY",-118.243344,34.052238
57,"Fresno, CA",-119.785856,36.731654
60,"Fargo, ND",-119.785856,36.731654
132,"Sanford, FL",-122.420168,37.780080
11,"San Francisco, CA (Metropolitan Area)",-122.420168,37.780080
20,"Portland, OR",-122.681425,45.516018
84,"Portland, ME",-122.681425,45.516018
157,"Wilmington, DE",-75.545100,39.744432
154,"Wilmington, NC",-75.545100,39.744432


**2. Is every year covered?** No. Full coverage through 2018, half in 2019-2020, none from 2021.

In [17]:
table1.groupby("Year")["Geocoded_City1"].apply(lambda s: s.notna().mean())

,Geocoded_City1
Year,
1996,1.000000
1997,1.000000
1998,1.000000
1999,1.000000
2000,1.000000
2001,1.000000
2002,1.000000
2003,1.000000
2004,1.000000


**3. Do some cities have no coordinates at all?** Yes, 11.

In [18]:
# 178 cities in the data but only 167 have coordinates
missing_geo = sorted(all_cities - set(geo["city"]))

print(len(missing_geo), "cities have no geocode at all")
missing_geo

11 cities have no geocode at all


['Ashland, WV',
 'Charlottesville, VA',
 'Hilton Head, SC',
 'Montrose/Delta, CO',
 'New Haven, CT',
 'Quad Cities, IL (Metropolitan Area)',
 'Springfield, IL',
 'St. Cloud, MN',
 'Traverse City, MI',
 'Vero Beach, FL',
 'Worcester, MA']

**4. Blank columns, and columns that repeat city1/city2**

In [19]:
# columns that are blank in every single row
table1.columns[table1.isna().all()].tolist()

['Geocoded_City1 (address)',
 'Geocoded_City1 (state)',
 'Geocoded_City1 (zip)',
 'Geocoded_City2 (address)',
 'Geocoded_City2 (state)',
 'Geocoded_City2 (zip)']

In [20]:
# the "(city)" columns are the same as city1 / city2
mask1 = table1["Geocoded_City1 (city)"].notna()
mask2 = table1["Geocoded_City2 (city)"].notna()
same1 = (table1.loc[mask1, "Geocoded_City1 (city)"] == table1.loc[mask1, "city1"]).all()
same2 = (table1.loc[mask2, "Geocoded_City2 (city)"] == table1.loc[mask2, "city2"]).all()
same1, same2

np.True_

### Drop every geocode column

In [21]:
# Drop every geocode column
df = table1.drop(columns=[c for c in table1.columns if c.startswith("Geocoded")])
df.shape

(120035, 16)

## Step 3. Rebuild coordinates from the BTS Master Coordinate table

Source: BTS Aviation Support Tables, Master Coordinate (TranStats).
Both tables use CityMarketID, so the join uses that key instead of city names.

In [22]:
# BTS Aviation Support Tables - Master Coordinate
MASTER_PATH = "/content/drive/MyDrive/770 EDA_Team 9/airport_master.csv"
master = pd.read_csv(MASTER_PATH, dtype=str)

master.shape

(20283, 32)

In [23]:
# this table keeps a row per airport per time period, and includes foreign airports
# IS_LATEST keeps only the current attributes for each airport
us = master[
    (master["AIRPORT_IS_LATEST"] == "1")
    & (master["AIRPORT_IS_CLOSED"] == "0")
    & (master["AIRPORT_COUNTRY_CODE_ISO"] == "US")
].copy()

print(len(us), "current US airports")

2830 current US airports


In [24]:
# citymarketid_1 has thousands separators, citymarketid_2 does not
# both tables use the same DOT CityMarketID
our_ids = set(df["citymarketid_1"].str.replace(",", "")) | set(df["citymarketid_2"])
bts_ids = set(us["CITY_MARKET_ID"])

print(len(our_ids), "market ids in our data")
print(len(our_ids & bts_ids), "of them found in BTS master")

178 market ids in our data
178 of them found in BTS master


Airports that belong to the 178 city markets

In [25]:
# stored as text, need floats to compare and average them
us["LATITUDE"] = us["LATITUDE"].astype(float)
us["LONGITUDE"] = us["LONGITUDE"].astype(float)

# keep only the markets that appear in the airfare data
mine = us[us["CITY_MARKET_ID"].isin(our_ids)].copy()
len(mine)

415

Group the 415 airports into the 178 city markets

In [26]:
# a city market can be served by several airports, so they need collapsing to one point
counts = mine.groupby("CITY_MARKET_ID").size()

print((counts > 1).sum(), "of 178 markets have multiple airports")
counts.value_counts().sort_index()

85 of 178 markets have multiple airports


,count
1,93
2,35
3,19
4,13
5,3
6,3
7,4
8,4
10,1
11,1


Some markets have more than one airport (New York has 15). The data is at the market level, so each market gets one point: the average of its airports. At U.S. map scale the difference from any single airport is small.

In [27]:
# average the airports in each market into a single point
market_geo = (
    mine.groupby("CITY_MARKET_ID")[["LATITUDE", "LONGITUDE"]].mean().reset_index()
)
market_geo.columns = ["market_id", "lat", "lon"]

print(len(market_geo), "markets with coordinates")
market_geo.head()

178 markets with coordinates


,market_id,lat,lon
0,30135,40.651667,-75.442778
1,30140,35.091528,-106.702639
2,30154,41.253333,-70.060556
3,30157,40.890556,-124.110556
4,30158,39.459167,-74.578611


Check the rebuilt coordinates:
1. No two markets share a point
2. Every latitude is inside the contiguous U.S. (24 to 50)
3. Every longitude is inside the contiguous U.S. (-125 to -66)

In [28]:
# run the same checks the original geocode columns failed
print("duplicate coordinates:", market_geo.duplicated(["lat", "lon"]).sum())
print("outside lat 24-50:", (~market_geo["lat"].between(24, 50)).sum())
print("outside lon -125 to -66:", (~market_geo["lon"].between(-125, -66)).sum())

duplicate coordinates: 0
outside lat 24-50: 0
outside lon -125 to -66: 0


In [29]:
# strip the separators so the ids match the BTS format
df["citymarketid_1"] = df["citymarketid_1"].str.replace(",", "")

# each route has two endpoints, so join twice
df = df.merge(
    market_geo.add_suffix("_1"),
    left_on="citymarketid_1",
    right_on="market_id_1",
    how="left",
)
df = df.merge(
    market_geo.add_suffix("_2"),
    left_on="citymarketid_2",
    right_on="market_id_2",
    how="left",
)

print("missing lat_1:", df["lat_1"].isna().sum())
print("missing lat_2:", df["lat_2"].isna().sum())
df.shape

missing lat_1: 0
missing lat_2: 0


(120035, 22)

In [30]:
# join keys are duplicates of the citymarketid columns
df = df.drop(columns=["market_id_1", "market_id_2"])
df.shape

(120035, 20)

In [32]:
# does the filter actually remove anything from our 178 markets
all_rows = master[master['CITY_MARKET_ID'].isin(our_ids)]

print('rows before filter:', len(all_rows))
print('rows after filter:', len(mine))
print()
print('IS_LATEST = 0 :', (all_rows['AIRPORT_IS_LATEST'] == '0').sum())
print('IS_CLOSED = 1 :', (all_rows['AIRPORT_IS_CLOSED'] == '1').sum())

rows before filter: 1465
rows after filter: 415

IS_LATEST = 0 : 1011
IS_CLOSED = 1 : 47


Sensitivity check: compare against coordinates built without the filter. 110 markets move, by at most 8 km, and every point stays inside its own metro area. The filter choice does not change the result.

In [33]:
# rebuild without the filter to see if the averages actually differ
unf = master.copy()
unf['LATITUDE'] = unf['LATITUDE'].astype(float)
unf['LONGITUDE'] = unf['LONGITUDE'].astype(float)
theirs = unf.groupby('CITY_MARKET_ID')[['LATITUDE', 'LONGITUDE']].mean()

both = market_geo.set_index('market_id').join(theirs, how='inner')
gap = (both['lat'] - both['LATITUDE']).abs()

print('markets that differ:', (gap > 0.001).sum(), 'of', len(both))
print('largest gap in degrees:', round(gap.max(), 4))

markets that differ: 110 of 178
largest gap in degrees: 0.0724


## Step 4. Clean data types

Convert money, counts, and shares to numbers.

In [ ]:
df.info()

In [ ]:
# money columns: strip "$" and "," then convert
for col in ["fare", "fare_lg", "fare_low"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")

In [ ]:
# passengers: strip "," (3 blank rows stay NaN)
df["passengers"] = pd.to_numeric(df["passengers"].astype(str).str.replace(",", "", regex=False), errors="coerce")

In [ ]:
# market shares (lf_ms has 3 blanks) and distance
df["large_ms"] = pd.to_numeric(df["large_ms"], errors="coerce")
df["lf_ms"]    = pd.to_numeric(df["lf_ms"], errors="coerce")
df["nsmiles"]  = pd.to_numeric(df["nsmiles"], errors="coerce").astype("Int64")

In [38]:
# whole-number columns
for col in ["Year", "quarter", "table_1_flag"]:
    df[col] = df[col].astype(int)

# coordinates set on float
for col in ["lat_1", "lon_1", "lat_2", "lon_2"]:
    df[col] = df[col].astype(float)

# citymarketid_1/2 stay as text ids
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120035 entries, 0 to 120034
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Year            120035 non-null  int64  
 1   quarter         120035 non-null  int64  
 2   citymarketid_1  120035 non-null  object 
 3   citymarketid_2  120035 non-null  object 
 4   city1           120035 non-null  object 
 5   city2           120035 non-null  object 
 6   nsmiles         120035 non-null  Int64  
 7   passengers      120035 non-null  int64  
 8   fare            120035 non-null  float64
 9   carrier_lg      120035 non-null  object 
 10  large_ms        120035 non-null  float64
 11  fare_lg         120035 non-null  float64
 12  carrier_low     120032 non-null  object 
 13  lf_ms           120032 non-null  float64
 14  fare_low        120032 non-null  float64
 15  table_1_flag    120035 non-null  int64  
 16  lat_1           120035 non-null  float64
 17  lon_1     

In [39]:
# the only NaNs should be the 3 known blank fare_low / lf_ms rows
print("nulls per column:")
print(df.isna().sum()[lambda s: s > 0])
df[["fare", "passengers", "large_ms", "nsmiles"]].describe()

nulls per column:
carrier_low    3
lf_ms          3
fare_low       3
dtype: int64


,fare,passengers,large_ms,nsmiles
count,120035.000000,120035.000000,120035.000000,120035.0
mean,199.963576,826.568984,0.557047,1057.003532
std,62.882462,1317.266929,0.184332,609.257641
min,56.420000,28.000000,0.020000,109.0
25%,154.780000,243.000000,0.415400,587.0
50%,194.560000,415.000000,0.540000,929.0
75%,240.350000,845.000000,0.680000,1433.0
max,676.890000,25471.000000,1.000000,2724.0


## Step 5. Time field

One quarterly field (1996Q1 to 2025Q4) plus a quarter-start date for plotting, sorted oldest to newest.

In [40]:
# one quarterly period, e.g. 1996Q1 ... 2025Q4
df["year_quarter"] = pd.PeriodIndex(
    df["Year"].astype(str) + "Q" + df["quarter"].astype(str), freq="Q"
)
# a real timestamp at the start of each quarter
df["quarter_start"] = df["year_quarter"].dt.to_timestamp()

# chronological order
df = df.sort_values(["year_quarter", "citymarketid_1", "citymarketid_2"]).reset_index(drop=True)
print(df["year_quarter"].min(), "->", df["year_quarter"].max(),
      "|", df["year_quarter"].nunique(), "quarters")
df[["Year", "quarter", "year_quarter", "quarter_start"]].head()

1996Q1 -> 2025Q4 | 120 quarters


,Year,quarter,year_quarter,quarter_start
0,1996,1,1996Q1,1996-01-01
1,1996,1,1996Q1,1996-01-01
2,1996,1,1996Q1,1996-01-01
3,1996,1,1996Q1,1996-01-01
4,1996,1,1996Q1,1996-01-01


## Step 6. Carrier names

Map the 2-letter carrier codes to airline names and add a low-cost carrier flag.

RU is marked as Britt Airways ("likely"). The code is not in the current BTS table because it was later reassigned, so it was identified from its route pattern (Cleveland, Houston, Newark hubs, 1996-97 only).

In [41]:
carriers = pd.read_csv("/content/drive/MyDrive/770 EDA_Team 9/carrier_lookup.csv",
                       dtype={"carrier_code": str})

# RU: not in the current BTS table, identified from route pattern
carriers.loc[carriers["carrier_code"] == "RU",
             ["airline_name", "confidence"]] = ["Britt Airways", "likely"]

# drop old name columns so rerunning this cell does not duplicate them
df = df.drop(columns=[c for c in ["carrier_lg_name", "lg_is_lcc",
                                  "carrier_low_name", "low_is_lcc"] if c in df.columns])

# name + LCC flag for the largest-share carrier, then for the lowest-fare carrier
for code_col, name_col, lcc_col in [("carrier_lg", "carrier_lg_name", "lg_is_lcc"),
                                    ("carrier_low", "carrier_low_name", "low_is_lcc")]:
    df = df.merge(
        carriers[["carrier_code", "airline_name", "is_lcc"]]
            .rename(columns={"airline_name": name_col, "is_lcc": lcc_col}),
        left_on=code_col, right_on="carrier_code", how="left").drop(columns="carrier_code")

print("unmatched carrier_lg :", df["carrier_lg_name"].isna().sum())
print("unmatched carrier_low:", df["carrier_low_name"].isna().sum())
df[df["carrier_lg"] == "RU"][["carrier_lg", "carrier_lg_name"]].head(1)

unmatched carrier_lg : 0
unmatched carrier_low: 3


,carrier_lg,carrier_lg_name
1253,RU,Britt Airways


The 3 unmatched carrier_low rows are blank in the source data. They stay in the dataset; drop them only for analysis that needs carrier_low.

In [42]:
OUT = "/content/drive/MyDrive/770 EDA_Team 9/airfare_clean.csv"
df.to_csv(OUT, index=False)
print("saved", df.shape, "to", OUT)
df.head()

saved (120035, 26) to /content/drive/MyDrive/770 EDA_Team 9/airfare_clean.csv


,Year,quarter,citymarketid_1,citymarketid_2,city1,city2,nsmiles,passengers,fare,carrier_lg,...,lat_1,lon_1,lat_2,lon_2,year_quarter,quarter_start,carrier_lg_name,lg_is_lcc,carrier_low_name,low_is_lcc
0,1996,1,30135,30977,"Allentown/Bethlehem/Easton, PA","Chicago, IL",654,113,274.24,UA,...,40.651667,-75.442778,41.843924,-87.835556,1996Q1,1996-01-01,United Airlines,0,Northwest Airlines,0.0
1,1996,1,30135,31454,"Allentown/Bethlehem/Easton, PA","Orlando, FL",906,235,198.67,FL,...,40.651667,-75.442778,28.421481,-81.358889,1996Q1,1996-01-01,AirTran Airways,1,US Airways,0.0
2,1996,1,30140,30194,"Albuquerque, NM","Dallas/Fort Worth, TX",580,1061,89.76,WN,...,35.091528,-106.702639,32.868889,-97.060732,1996Q1,1996-01-01,Southwest Airlines,1,Southwest Airlines,1.0
3,1996,1,30140,30325,"Albuquerque, NM","Denver, CO",349,345,143.52,UA,...,35.091528,-106.702639,39.729236,-104.702292,1996Q1,1996-01-01,United Airlines,0,Frontier Airlines,1.0
4,1996,1,30140,30397,"Albuquerque, NM","Atlanta, GA (Metropolitan Area)",1269,142,178.04,DL,...,35.091528,-106.702639,33.737257,-84.457361,1996Q1,1996-01-01,Delta Air Lines,0,Delta Air Lines,0.0


In [43]:
# final check before moving to analysis
print(df.shape)
print()
print("nulls:", df.isna().sum()[lambda s: s > 0].to_dict())
print("fare range:", df["fare"].min(), "-", df["fare"].max())
print("coordinate nulls:", df[["lat_1", "lon_1", "lat_2", "lon_2"]].isna().sum().sum())

(120035, 26)

nulls: {'carrier_low': 3, 'lf_ms': 3, 'fare_low': 3, 'carrier_low_name': 3, 'low_is_lcc': 3}
fare range: 56.42 - 676.89
coordinate nulls: 0


## Step 7. External data

Two outside factors, to test whether they explain fares:
1. **Market size**: population within 40 miles of each city market (SimpleMaps / U.S. Census ZIP data)
2. **Market wealth**: median household income of each endpoint state (Census ACS 2023)

In [47]:
# population within 40 mi of each city market
pop = pd.read_csv("/content/drive/MyDrive/770 EDA_Team 9/market_population.csv",
                  dtype={"market_id": str})
pm = dict(zip(pop["market_id"], pop["catchment_pop_40mi"]))

df["pop1"] = df["citymarketid_1"].astype(str).map(pm)
df["pop2"] = df["citymarketid_2"].astype(str).map(pm)
print("population matched:", df["pop1"].notna().mean().round(3), df["pop2"].notna().mean().round(3))

population matched: 1.0 1.0


In [48]:
import re
# U.S. Census ACS 2023 1-year, median household income by state
income = {'DC':108210,'MA':99858,'NJ':99781,'MD':98678,'NH':96838,'CA':95521,'HI':95322,'WA':94605,
 'UT':93421,'CO':92911,'CT':91665,'VA':89931,'AK':88121,'MN':85086,'RI':84972,'DE':82174,'NY':82095,
 'VT':81211,'IL':80306,'OR':80160,'AZ':77315,'ND':76525,'NV':76364,'TX':75780,'ID':74942,'GA':74632,
 'WI':74631,'NE':74590,'PA':73824,'ME':73733,'FL':73311,'WY':72415,'SD':71810,'IA':71433,'MT':70804,
 'NC':70804,'KS':70333,'IN':69477,'MI':69183,'MO':68545,'SC':67804,'OH':67769,'TN':67631,'NM':62268,
 'AL':62212,'OK':62138,'KY':61118,'AR':58700,'LA':58229,'WV':55948,'MS':54203}

# the state sits inside the city name, e.g. "Miami, FL (Metropolitan Area)" -> FL
get_state = lambda s: (re.search(r",\s*([A-Z]{2})\b", str(s)) or [None, None])[1]
df["state1"] = df["city1"].map(get_state)
df["state2"] = df["city2"].map(get_state)
df["income1"] = df["state1"].map(income)
df["income2"] = df["state2"].map(income)
print("income matched:", df["income1"].notna().mean().round(3))
df[["city1","state1","pop1","income1"]].head()

income matched: 1.0


,city1,state1,pop1,income1
0,"Allentown/Bethlehem/Easton, PA",PA,2872982.0,73824
1,"Allentown/Bethlehem/Easton, PA",PA,2872982.0,73824
2,"Albuquerque, NM",NM,910492.0,62268
3,"Albuquerque, NM",NM,910492.0,62268
4,"Albuquerque, NM",NM,910492.0,62268


**Result:** neither closes the gap. Income adds under 1% to R-squared. Population does not predict fare directly, but larger metros have a smaller cheap-airline share.

In [49]:
OUT = "/content/drive/MyDrive/770 EDA_Team 9/airfare_enriched.csv"
df.to_csv(OUT, index=False)
print("saved", df.shape, "to", OUT)
df.head()

saved (120035, 32) to /content/drive/MyDrive/770 EDA_Team 9/airfare_enriched.csv


,Year,quarter,citymarketid_1,citymarketid_2,city1,city2,nsmiles,passengers,fare,carrier_lg,...,carrier_lg_name,lg_is_lcc,carrier_low_name,low_is_lcc,pop1,pop2,state1,state2,income1,income2
0,1996,1,30135,30977,"Allentown/Bethlehem/Easton, PA","Chicago, IL",654,113,274.24,UA,...,United Airlines,0,Northwest Airlines,0.0,2872982.0,8682366.0,PA,IL,73824,80306
1,1996,1,30135,31454,"Allentown/Bethlehem/Easton, PA","Orlando, FL",906,235,198.67,FL,...,AirTran Airways,1,US Airways,0.0,2872982.0,3484502.0,PA,FL,73824,73311
2,1996,1,30140,30194,"Albuquerque, NM","Dallas/Fort Worth, TX",580,1061,89.76,WN,...,Southwest Airlines,1,Southwest Airlines,1.0,910492.0,7472864.0,NM,TX,62268,75780
3,1996,1,30140,30325,"Albuquerque, NM","Denver, CO",349,345,143.52,UA,...,United Airlines,0,Frontier Airlines,1.0,910492.0,3346564.0,NM,CO,62268,92911
4,1996,1,30140,30397,"Albuquerque, NM","Atlanta, GA (Metropolitan Area)",1269,142,178.04,DL,...,Delta Air Lines,0,Delta Air Lines,0.0,910492.0,5728934.0,NM,GA,62268,74632
